<a href="https://colab.research.google.com/github/harshav6/deepLearning/blob/main/perceptron12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
iris = load_iris()

In [ ]:
X = iris.data
y = iris.target

In [ ]:
Y = np.eye(3)[y]

In [ ]:
df = pd.DataFrame(iris.data, columns = iris.feature_names)

In [ ]:
df["target"] = iris.target

In [ ]:
df["species"] = df["target"].map({
    0: "setosa",
    1: "versicolor",
    2: "virginica"
})

In [ ]:
print(df.head())

   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

   target species  
0       0  setosa  
1       0  setosa  
2       0  setosa  
3       0  setosa  
4       0  setosa  


In [ ]:
print(df.shape)

(150, 6)


In [ ]:
df.columns

Index(['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)',
       'petal width (cm)', 'target', 'species'],
      dtype='object')

In [ ]:
df.describe()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
count,150.000000,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.057333,3.758000,1.199333,1.000000
std,0.828066,0.435866,1.765298,0.762238,0.819232
min,4.300000,2.000000,1.000000,0.100000,0.000000
25%,5.100000,2.800000,1.600000,0.300000,0.000000
50%,5.800000,3.000000,4.350000,1.300000,1.000000
75%,6.400000,3.300000,5.100000,1.800000,2.000000
max,7.900000,4.400000,6.900000,2.500000,2.000000


In [ ]:
print(df.isnull().sum())

sepal length (cm)    0
sepal width (cm)     0
petal length (cm)    0
petal width (cm)     0
target               0
species              0
dtype: int64


In [ ]:
print(df.duplicated().sum())

1


In [ ]:
print(df["species"].unique())

['setosa' 'versicolor' 'virginica']


In [ ]:
print(df["species"].value_counts())

species
setosa        50
versicolor    50
virginica     50
Name: count, dtype: int64


In [ ]:
# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42
)

In [ ]:
# Feature Scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
class NeuralNetwork:
  def __init__(self, input_size, hidden1_size,hidden2_size, output_size, learning_rate=0.01):
    self.input_size = input_size
    self.hidden1_size = hidden1_size
    self.hidden2_size = hidden2_size
    self.output_size = output_size
    self.learning_rate = 0.01

    self.W1 = np.random.randn(input_size, hidden1_size) * np.sqrt(2/input_size)
    self.b1 = np.zeros((1, hidden1_size))

    self.W2 = np.random.randn(hidden1_size, hidden2_size) * np.sqrt(2/hidden1_size)
    self.b2 = np.zeros((1, hidden2_size))

    self.W3 = np.random.randn(hidden2_size, output_size) * np.sqrt(2/hidden2_size)
    self.b3 = np.zeros((1, output_size))

  def relu(self, x):
    return np.maximum(0, x)

  def relu_derivative(self, x):
    return (x > 0).astype(float)

  def softmax(self, x):
     exp = np.exp(x - np.max(x, axis=1, keepdims=True))
     return exp / np.sum(exp, axis=1, keepdims=True)

  def forward(self, X):
    self.X = X
    self.z1 = np.dot(X, self.W1) + self.b1
    self.a1 = self.relu(self.z1)

    self.z2 = np.dot(self.a1, self.W2) + self.b2
    self.a2 = self.softmax(self.z2)

    self.z3 = np.dot(self.a2, self.W3) + self.b3
    self.a3 = self.softmax(self.z3)

    return self.a3

  def backward(self, y):
    m = y.shape[0]

        # Output Layer
    dz3 = self.a3 - y

    dW3 = np.dot(self.a2.T, dz3) / m
    db3 = np.sum(dz3, axis=0, keepdims=True) / m

        # Hidden Layer 2
    da2 = np.dot(dz3, self.W3.T)
    dz2 = da2 * self.relu_derivative(self.z2)

    dW2 = np.dot(self.a1.T, dz2) / m
    db2 = np.sum(dz2, axis=0, keepdims=True) / m

        # Hidden Layer 1
    da1 = np.dot(dz2, self.W2.T)
    dz1 = da1 * self.relu_derivative(self.z1)

    dW1 = np.dot(self.X.T, dz1) / m
    db1 = np.sum(dz1, axis=0, keepdims=True) / m

        # Update Parameters
    self.W3 -= self.learning_rate * dW3
    self.b3 -= self.learning_rate * db3

    self.W2 -= self.learning_rate * dW2
    self.b2 -= self.learning_rate * db2

    self.W1 -= self.learning_rate * dW1
    self.b1 -= self.learning_rate * db1

  def loss(self, y):
    m = y.shape[0]
    return -np.sum(y * np.log(self.a3 + 1e-8)) / m

  def predict(self, X):
    output = self.forward(X)
    return np.argmax(output, axis=1)


In [ ]:
nn = NeuralNetwork(
    input_size=4,
    hidden1_size=8,
    hidden2_size=8,
    output_size=3,
    learning_rate=0.1
)

epochs = 1000

for epoch in range(epochs):

    nn.forward(X_train)

    nn.backward(y_train)

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss = {nn.loss(y_train):.4f}")

pred = nn.predict(X_test)

actual = np.argmax(y_test, axis=1)

accuracy = np.mean(pred == actual)

print("\nAccuracy:", accuracy)

print(self.a1.min(), self.a1.max())
print(self.a2.min(), self.a2.max())

Epoch 0, Loss = 1.1365
Epoch 100, Loss = 1.0909
Epoch 200, Loss = 1.0555
Epoch 300, Loss = 1.0031
Epoch 400, Loss = 0.9343
Epoch 500, Loss = 0.8433
Epoch 600, Loss = 0.7432
Epoch 700, Loss = 0.6654
Epoch 800, Loss = 0.6146
Epoch 900, Loss = 0.5789

Accuracy: 0.8666666666666667


NameError: name 'self' is not defined